In [ ]:
# Core Python
import os
import re
import json
import math
import random
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

np.random.seed(42)
random.seed(42)

DATA_DIR = Path(".")
EMB_DIR = DATA_DIR / "embeddings"
EMB_DIR.mkdir(exist_ok=True)


In [ ]:
# --- Part 1.1: Term–document matrix + TF-IDF (article-level) ---

MAX_VOCAB = 10000
UNK = "<UNK>"


def parse_articles(path: Path):
    """Split cleaned.txt into articles using [id] markers."""
    lines = path.read_text(encoding="utf-8").splitlines()
    articles = {}
    cur_id = None
    buf = []
    for raw in lines:
        m = re.match(r"^\s*\[(\d+)\]\s*$", raw)
        if m:
            if cur_id is not None:
                articles[cur_id] = "\n".join(buf)
            cur_id = int(m.group(1))
            buf = []
        else:
            buf.append(raw)
    if cur_id is not None:
        articles[cur_id] = "\n".join(buf)
    return articles


def tokenize(text: str):
    return text.split()


CATEGORY_NAMES = {
    0: "Other",
    1: "Politics",
    2: "Sports",
    3: "Economy",
    4: "International",
    5: "Health & Society",
}

KEYWORDS = {
    1: [
        "election", "government", "minister", "parliament",
        "وزیر", "حکومت", "انتخاب", "پارلیمنٹ", "صدر", "وزارت",
        "مسلم لیگ", "پی ٹی آئی", "اپوزیشن", "سیاست",
    ],
    2: [
        "cricket", "match", "team", "player", "score",
        "کرکٹ", "میچ", "کھلاڑی", "ٹیم", "پی ایس ایل", "پی سی بی",
        "ورلڈ کپ", "اسپورٹس", "سپورٹس", "فٹبال", "ہاکی",
    ],
    3: [
        "inflation", "trade", "bank", "gdp", "budget",
        "معیشت", "روپیہ", "ڈالر", "بجٹ", "بینک", "کاروبار",
        "برآمد", "درآمد", "مارکیٹ", "اسٹاک",
    ],
    4: [
        "treaty", "foreign", "bilateral", "conflict",
        "بین الاقوامی", "امریکہ", "چین", "افغانستان", "اسرائیل",
        "iran", "فلسطین", "اقوام متحدہ", "مذاکرات", "جنگ",
    ],
    5: [
        "hospital", "disease", "vaccine", "flood", "education",
        "صحت", "بیماری", "اسپتال", "ویکسین", "سیلاب", "تعلیم",
        "اسکول", "یونیورسٹی", "ڈاکٹر", "علاج",
    ],
}


def assign_category(text: str, title: str) -> int:
    blob = (title + " " + text[:8000]).lower()
    scores = {cid: 0 for cid in KEYWORDS}
    for cid, words in KEYWORDS.items():
        for w in words:
            if w.lower() in blob or w in blob:
                scores[cid] += 1
    best_c = max(scores, key=scores.get)
    return best_c if scores[best_c] > 0 else 0


articles_raw = parse_articles(DATA_DIR / "cleaned.txt")
meta = json.loads((DATA_DIR / "articles_metadata.json").read_text(encoding="utf-8"))

article_ids = sorted(articles_raw.keys())
article_texts = [articles_raw[i] for i in article_ids]
article_tokens = [tokenize(t) for t in article_texts]
titles = [meta.get(str(i), {}).get("title", "") for i in article_ids]
article_categories = [assign_category(articles_raw[i], titles[j]) for j, i in enumerate(article_ids)]

token_counts = Counter()
for toks in article_tokens:
    token_counts.update(toks)

most_common = token_counts.most_common(MAX_VOCAB)
vocab = {UNK: 0}
for idx, (tok, _) in enumerate(most_common, start=1):
    vocab[tok] = idx
idx_to_token = {i: t for t, i in vocab.items()}
V = len(vocab)

def tokens_to_ids(toks):
    return [vocab.get(t, 0) for t in toks]

num_docs = len(article_tokens)
td_matrix = np.zeros((num_docs, V), dtype=np.int32)
for di, toks in enumerate(article_tokens):
    for tid in tokens_to_ids(toks):
        td_matrix[di, tid] += 1

N_docs = td_matrix.shape[0]
df = np.sum(td_matrix > 0, axis=0)
idf = np.log(N_docs / (1.0 + df))
tf = td_matrix.astype(np.float64)
tfidf_matrix = tf * idf

np.save(EMB_DIR / "tfidf_matrix.npy", tfidf_matrix.astype(np.float32))
print("Saved", EMB_DIR / "tfidf_matrix.npy", "shape", tfidf_matrix.shape)

print("\nTop-10 discriminative words per topic category (mean TF-IDF within category, excluding <UNK>):")
for cat in range(1, 6):
    mask = np.array([c == cat for c in article_categories], dtype=bool)
    if not np.any(mask):
        print(f"\n{CATEGORY_NAMES[cat]}: (no documents)")
        continue
    mean_vec = tfidf_matrix[mask].mean(axis=0)
    mean_vec[0] = -1.0
    top_idx = np.argsort(mean_vec)[::-1][:10]
    words = [idx_to_token[i] for i in top_idx]
    print(f"\n{CATEGORY_NAMES[cat]}:")
    print(", ".join(words))

print("\nVocabulary size (incl. <UNK>):", V)


In [ ]:
# --- Part 1.2: PPMI co-occurrence (symmetric window k=5), t-SNE, neighbours ---

WINDOW_K = 5

co_counts = np.zeros((V, V), dtype=np.float64)
for toks in article_tokens:
    ids = tokens_to_ids(toks)
    n = len(ids)
    for i in range(n):
        lo = max(0, i - WINDOW_K)
        hi = min(n, i + WINDOW_K + 1)
        for j in range(lo, hi):
            if i == j:
                continue
            wi, wj = ids[i], ids[j]
            co_counts[wi, wj] += 1.0

total = co_counts.sum()
row_sum = co_counts.sum(axis=1)
col_sum = co_counts.sum(axis=0)
eps = 1e-10
Pwc = co_counts / (total + eps)
Pw = row_sum / (total + eps)
Pcontext = col_sum / (total + eps)
pmi = np.log2((Pwc + eps) / (Pw[:, None] * Pcontext[None, :] + eps))
ppmi_matrix = np.maximum(0.0, pmi).astype(np.float32)
np.save(EMB_DIR / "ppmi_matrix.npy", ppmi_matrix)
print("Saved", EMB_DIR / "ppmi_matrix.npy", "shape", ppmi_matrix.shape)

# --- t-SNE on 200 most frequent tokens (rows of PPMI), colour by seed-based semantic bucket ---

freq_token_ids = [vocab[tok] for tok, _ in most_common[:200]]

SEED_BY_CAT = {
    "Politics": ["حکومت", "وزیر", "پارلیمنٹ", "انتخابات", "صوبائی حکومت"],
    "Sports": ["کرکٹ", "میچ", "کھلاڑی", "ٹیم", "پی ایس ایل"],
    "Geography": ["کراچی", "لاہور", "اسلام آباد", "پشاور", "کوئٹہ", "سندھ", "پنجاب"],
    "Economy": ["معیشت", "بجٹ", "بینک", "روپیہ", "ڈالر"],
    "International": ["امریکہ", "چین", "افغانستان", "اقوام متحدہ", "جنگ"],
}

seed_indices = {lab: [] for lab in SEED_BY_CAT}
for lab, seeds in SEED_BY_CAT.items():
    for s in seeds:
        if s in vocab:
            seed_indices[lab].append(vocab[s])

sub = ppmi_matrix[np.ix_(freq_token_ids, freq_token_ids)].astype(np.float64)
row_norm = np.linalg.norm(sub, axis=1, keepdims=True) + 1e-9
sub_n = sub / row_norm

labels_200 = []
scores_debug = []
for row_i, wid in enumerate(freq_token_ids):
    best_lab = "Other"
    best_score = -1.0
    for lab, sidxs in seed_indices.items():
        if not sidxs:
            continue
        sc = float(ppmi_matrix[wid, sidxs].sum())
        if sc > best_score:
            best_score = sc
            best_lab = lab
    if best_score <= 0:
        best_lab = "Other"
    labels_200.append(best_lab)

perplexity = min(30, max(5, len(freq_token_ids) // 4))
tsne = TSNE(n_components=2, random_state=42, perplexity=perplexity, init="pca", learning_rate="auto")
coords = tsne.fit_transform(sub_n)

plt.figure(figsize=(12, 8))
unique_labs = sorted(set(labels_200))
colors = plt.cm.tab10(np.linspace(0, 1, max(len(unique_labs), 1)))
lab_to_color = {lab: colors[i % len(colors)] for i, lab in enumerate(unique_labs)}
for lab in unique_labs:
    idxs = [i for i, l in enumerate(labels_200) if l == lab]
    plt.scatter(
        coords[idxs, 0],
        coords[idxs, 1],
        c=[lab_to_color[lab]],
        label=lab,
        s=22,
        alpha=0.85,
        edgecolors="none",
    )
plt.title("t-SNE of 200 most frequent tokens (PPMI rows, cosine-normalised)")
plt.xlabel("t-SNE dimension 1")
plt.ylabel("t-SNE dimension 2")
plt.legend(title="Semantic category", loc="best", fontsize=8)
plt.tight_layout()
plt.show()


def top_neighbors_ppmi(query: str, topk: int = 5):
    if query not in vocab:
        print(query, "not in vocabulary")
        return
    qi = vocab[query]
    v = ppmi_matrix[qi].astype(np.float64)
    nv = np.linalg.norm(v) + 1e-9
    sims = (ppmi_matrix.astype(np.float64) @ v) / (np.linalg.norm(ppmi_matrix.astype(np.float64), axis=1) * nv + 1e-9)
    sims[qi] = -1e9
    best = np.argsort(sims)[::-1][:topk]
    out = [(idx_to_token[int(j)], float(sims[j])) for j in best]
    return out


PPMI_QUERY_WORDS = [
    "پاکستان", "کرکٹ", "حکومت", "معیشت", "کراچی",
    "امریکہ", "تعلیم", "میچ", "وزیر", "سیلاب",
]
print("\nTop-5 PPMI cosine neighbours (>=10 query words):")
for w in PPMI_QUERY_WORDS:
    nbrs = top_neighbors_ppmi(w, 5)
    print(f"\n{w}:")
    if nbrs:
        for t, s in nbrs:
            print(f"  {t}\t{s:.4f}")


In [ ]:
# --- Part 2.1: Skip-gram Word2Vec (cleaned.txt, same 10K+UNK vocab) ---

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

EMBED_DIM = 100
WINDOW_SIZE = 5
NEG_SAMPLES = 10
LR = 0.001
BATCH_SIZE = 512
EPOCHS = 5


def build_skipgram_pairs(token_lists):
    pairs_local = []
    for ids in token_lists:
        n = len(ids)
        for i, c in enumerate(ids):
            lo = max(0, i - WINDOW_SIZE)
            hi = min(n, i + WINDOW_SIZE + 1)
            for j in range(lo, hi):
                if i == j:
                    continue
                pairs_local.append((c, ids[j]))
    return pairs_local


class SkipGramDataset(Dataset):
    def __init__(self, prs):
        self.prs = prs

    def __len__(self):
        return len(self.prs)

    def __getitem__(self, idx):
        c, o = self.prs[idx]
        return torch.tensor(c, dtype=torch.long), torch.tensor(o, dtype=torch.long)


class SkipGramNegSampling(nn.Module):
    def __init__(self, vocab_size, dim):
        super().__init__()
        self.V = nn.Embedding(vocab_size, dim)
        self.U = nn.Embedding(vocab_size, dim)
        nn.init.xavier_uniform_(self.V.weight)
        nn.init.xavier_uniform_(self.U.weight)

    def forward(self, center_words, pos_words, neg_words):
        vc = self.V(center_words)
        uo = self.U(pos_words)
        uk = self.U(neg_words)
        pos_score = torch.sum(vc * uo, dim=1)
        pos_loss = F.logsigmoid(pos_score)
        neg_score = torch.bmm(uk, vc.unsqueeze(2)).squeeze(2)
        neg_loss = F.logsigmoid(-neg_score).sum(dim=1)
        return -(pos_loss + neg_loss).mean()


def train_skipgram(token_lists, embed_dim, label):
    id_lists = [tokens_to_ids(t) for t in token_lists]
    prs = build_skipgram_pairs(id_lists)
    if not prs:
        raise ValueError("empty pairs")
    tf = np.zeros(V, dtype=np.float64)
    for ids in id_lists:
        for x in ids:
            tf[x] += 1.0
    nd = tf ** 0.75
    nd = nd / (nd.sum() + 1e-12)
    noise_t = torch.tensor(nd, dtype=torch.float32, device=device)
    print("Total training pairs:", label, len(prs))
    ds = SkipGramDataset(prs)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    mdl = SkipGramNegSampling(V, embed_dim).to(device)
    opt = optim.Adam(mdl.parameters(), lr=LR)
    loss_hist = []
    for epoch in range(EPOCHS):
        total = 0.0
        steps = 0
        pbar = tqdm(dl, desc=f"{label} ep{epoch+1}/{EPOCHS}")
        for centers, positives in pbar:
            centers = centers.to(device)
            positives = positives.to(device)
            negs = torch.multinomial(
                noise_t,
                centers.shape[0] * NEG_SAMPLES,
                replacement=True,
            ).view(centers.shape[0], NEG_SAMPLES)
            loss = mdl(centers, positives, negs)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item()
            steps += 1
            loss_hist.append(loss.item())
            pbar.set_postfix(loss=f"{loss.item():.4f}")
        print(f"{label} epoch {epoch+1} mean loss {total/max(steps,1):.4f}")
    with torch.no_grad():
        emb = 0.5 * (mdl.V.weight + mdl.U.weight).cpu().numpy()
    return emb, loss_hist


emb_c3, loss_c3 = train_skipgram(article_tokens, EMBED_DIM, "C3 cleaned d=100")
np.save(EMB_DIR / "embeddings_w2v.npy", emb_c3.astype(np.float32))
with open(EMB_DIR / "word2idx.json", "w", encoding="utf-8") as f:
    json.dump(vocab, f, ensure_ascii=False, indent=2)
print("Saved", EMB_DIR / "embeddings_w2v.npy", emb_c3.shape, EMB_DIR / "word2idx.json")

plt.figure(figsize=(10, 5))
plt.plot(loss_c3)
plt.title("Skip-gram training loss (C3 cleaned, d=100)")
plt.xlabel("Training step")
plt.ylabel("Loss")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# --- Part 2.2: Nearest neighbours, analogies, four-condition comparison + MRR ---


def normalize_rows(mat: np.ndarray) -> np.ndarray:
    n = np.linalg.norm(mat, axis=1, keepdims=True) + 1e-9
    return mat / n


def neighbours_w2v(query: str, emb: np.ndarray, topk: int = 10):
    if query not in vocab:
        print(query, "OOV")
        return
    qi = vocab[query]
    E = normalize_rows(emb.astype(np.float64))
    sims = E @ E[qi]
    sims[qi] = -1e9
    best = np.argsort(sims)[::-1][:topk]
    return [(idx_to_token[int(j)], float(sims[j])) for j in best]


EVAL_WORDS = [
    "Pakistan", "Hukumat", "Adalat", "Maeeshat", "Fauj", "Sehat", "Taleem", "Aabadi",
]
URDU_FALLBACK = {
    "Pakistan": "پاکستان",
    "Hukumat": "حکومت",
    "Adalat": "عدالت",
    "Maeeshat": "معیشت",
    "Fauj": "فوج",
    "Sehat": "صحت",
    "Taleem": "تعلیم",
    "Aabadi": "آبادی",
}

print("Top-10 nearest neighbours (C3 averaged embeddings):")
for w in EVAL_WORDS:
    key = w if w in vocab else URDU_FALLBACK.get(w, w)
    print(f"\n{w} (lookup: {key}):")
    nbs = neighbours_w2v(key, emb_c3, 10)
    if nbs:
        for t, s in nbs:
            print(f"  {t}\t{s:.4f}")


def analogy_vec(a, b, c, emb, topn=3):
    for x in (a, b, c):
        if x not in vocab:
            print("analogy OOV:", x)
            return
    E = normalize_rows(emb.astype(np.float64))
    va, vb, vc = E[vocab[a]], E[vocab[b]], E[vocab[c]]
    v = vb - va + vc
    v = v / (np.linalg.norm(v) + 1e-9)
    sims = E @ v
    sims[vocab[a]] = sims[vocab[b]] = sims[vocab[c]] = -1e9
    best = np.argsort(sims)[::-1][:topn]
    return [(idx_to_token[int(i)], float(sims[i])) for i in best]


ANALOGIES = [
    ("پاکستان", "کراچی", "سندھ"),
    ("پاکستان", "کراچی", "پنجاب"),
    ("حکومت", "وزیر", "عدالت"),
    ("کرکٹ", "میچ", "فٹبال"),
    ("تعلیم", "یونیورسٹی", "صحت"),
    ("صحت", "دوا", "تعلیم"),
    ("معیشت", "بجٹ", "حکومت"),
    ("فوج", "جنگ", "حکومت"),
    ("امریکہ", "ٹرمپ", "برطانیہ"),
    ("کراچی", "سندھ", "لاہور"),
]

print("\n10 analogy tests (a : b :: c : ?), top-3 by cosine:")
for a, b, c in ANALOGIES:
    print(f"\n{a} : {b} :: {c} : ?")
    res = analogy_vec(a, b, c, emb_c3, 3)
    if res:
        for t, s in res:
            print(f"  {t}\t{s:.4f}")

print(
    "\nShort assessment: neighbours often cluster by domain (politics, sports, cities)."
    " Vector arithmetic sometimes returns function words; larger data and longer training"
    " usually improve analogies for Urdu."
)

# --- Four conditions (C1–C4): top-5 neighbours for 5 queries + MRR on 20 labelled pairs ---

MRR_PAIRS = [
    ("پاکستان", "کراچی"),
    ("پاکستان", "لاہور"),
    ("حکومت", "وزیر"),
    ("حکومت", "عدالت"),
    ("کرکٹ", "میچ"),
    ("کرکٹ", "کھلاڑی"),
    ("تعلیم", "یونیورسٹی"),
    ("صحت", "دوا"),
    ("معیشت", "بجٹ"),
    ("بینک", "ڈالر"),
    ("فوج", "جنگ"),
    ("امریکہ", "ٹرمپ"),
    ("چین", "بیجنگ"),
    ("سندھ", "کراچی"),
    ("پنجاب", "لاہور"),
    ("اسرائیل", "فلسطین"),
    ("افغانستان", "طالبان"),
    ("بجٹ", "ٹیکس"),
    ("تعلیم", "طلبہ"),
    ("سیلاب", "متاثرہ"),
    ("برطانیہ", "لندن"),
]


def mean_reciprocal_rank(emb: np.ndarray, pairs):
    E = normalize_rows(emb.astype(np.float64))
    rrs = []
    for w, target in pairs:
        if w not in vocab or target not in vocab:
            continue
        wi, ti = vocab[w], vocab[target]
        sims = E @ E[wi]
        sims[wi] = -1e9
        order = np.argsort(sims)[::-1]
        rank = None
        for r, j in enumerate(order, start=1):
            if int(j) == ti:
                rank = r
                break
        rrs.append(1.0 / rank if rank else 0.0)
    return sum(rrs) / max(len(rrs), 1)


queries_5 = ["پاکستان", "کرکٹ", "حکومت", "معیشت", "تعلیم"]

print("\n--- Condition C1: PPMI rows (L2-normalised) ---")
E_c1 = normalize_rows(ppmi_matrix.astype(np.float64))
for w in queries_5:
    print(f"\n{w}:")
    if w not in vocab:
        continue
    qi = vocab[w]
    sims = E_c1 @ E_c1[qi]
    sims[qi] = -1e9
    for j in np.argsort(sims)[::-1][:5]:
        print(f"  {idx_to_token[int(j)]}\t{float(sims[j]):.4f}")
mrr_c1 = mean_reciprocal_rank(ppmi_matrix.astype(np.float64), MRR_PAIRS)
print("MRR (C1):", round(mrr_c1, 4))

print("\n--- Condition C2: Skip-gram on raw.txt (same vocab / UNK) ---")
articles_raw_bodies = parse_articles(DATA_DIR / "raw.txt")
raw_ids = sorted(articles_raw_bodies.keys())
raw_tokens = [tokenize(articles_raw_bodies[i]) for i in raw_ids]
emb_c2, _ = train_skipgram(raw_tokens, EMBED_DIM, "C2 raw d=100")
for w in queries_5:
    print(f"\n{w}:")
    nbs = neighbours_w2v(w, emb_c2, 5)
    if nbs:
        for t, s in nbs:
            print(f"  {t}\t{s:.4f}")
mrr_c2 = mean_reciprocal_rank(emb_c2, MRR_PAIRS)
print("MRR (C2):", round(mrr_c2, 4))

print("\n--- Condition C3: Skip-gram cleaned (already trained) ---")
for w in queries_5:
    print(f"\n{w}:")
    nbs = neighbours_w2v(w, emb_c3, 5)
    if nbs:
        for t, s in nbs:
            print(f"  {t}\t{s:.4f}")
mrr_c3 = mean_reciprocal_rank(emb_c3, MRR_PAIRS)
print("MRR (C3):", round(mrr_c3, 4))

print("\n--- Condition C4: Skip-gram cleaned d=200 ---")
emb_c4, _ = train_skipgram(article_tokens, 200, "C4 cleaned d=200")
for w in queries_5:
    print(f"\n{w}:")
    nbs = neighbours_w2v(w, emb_c4, 5)
    if nbs:
        for t, s in nbs:
            print(f"  {t}\t{s:.4f}")
mrr_c4 = mean_reciprocal_rank(emb_c4, MRR_PAIRS)
print("MRR (C4):", round(mrr_c4, 4))

print(
    "\nDiscussion: compare MRR across C1–C4. Skip-gram on cleaned text (C3/C4) typically"
    " beats raw noise (C2) and sparse high-dimensional PPMI (C1) on this metric; doubling d"
    " (C4) may help MRR slightly if the corpus is large enough to support the extra parameters."
)
